In [ ]:
# In[1]:

import pandas as pd
import pytz

# Load CSVs into memory (store full DataFrames for later use)
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_logs_df = pd.read_csv("error_logs.csv")

# Timezone (UTC+8)
tz = pytz.timezone('Asia/Shanghai')

# Helper to produce summary for metric-like files
def kpi_summary(df, cmdb_col, kpi_col, top_n=20):
    # ensure timestamp exists and compute tz-aware min/max
    grp = (
        df.groupby([cmdb_col, kpi_col], dropna=False)
          .agg(row_count=( 'timestamp', 'size'),
               min_ts=('timestamp', 'min'),
               max_ts=('timestamp', 'max'))
          .reset_index()
    )
    # convert min/max timestamps to tz-aware strings
    grp['min_timestamp'] = pd.to_datetime(grp['min_ts'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S%z')
    grp['max_timestamp'] = pd.to_datetime(grp['max_ts'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S%z')
    # rename KPI column uniformly
    grp = grp.rename(columns={kpi_col: 'KPI_name', cmdb_col: 'cmdb_id'})
    top = grp.sort_values('row_count', ascending=False).head(top_n)[['cmdb_id','KPI_name','row_count','min_timestamp','max_timestamp']].reset_index(drop=True)
    unique_cmdbs = sorted(df[cmdb_col].dropna().unique().tolist())
    unique_kpis = sorted(df[kpi_col].dropna().unique().tolist())
    total_rows = df.shape[0]
    distinct_pairs = grp.shape[0]
    return unique_cmdbs, unique_kpis, total_rows, distinct_pairs, top

# Metric summary
metric_unique_cmdbs, metric_unique_kpis, metric_total_rows, metric_distinct_pairs, metric_summary_top = kpi_summary(metric_df, 'cmdb_id', 'kpi_name', top_n=20)

# Trace summary
trace_unique_cmdbs, trace_unique_kpis, trace_total_rows, trace_distinct_pairs, trace_summary_top = kpi_summary(trace_df, 'cmdb_id', 'trace_name', top_n=20)

# Log summary
log_unique_cmdbs, log_unique_kpis, log_total_rows, log_distinct_pairs, log_summary_top = kpi_summary(log_df, 'cmdb_id', 'log_name', top_n=20)

# Error logs summary: total rows, unique cmdbs, top 5 rows by timestamp ascending
error_total_rows = error_logs_df.shape[0]
error_unique_cmdbs = sorted(error_logs_df['cmdb_id'].dropna().unique().tolist())

# Prepare top 5 rows sorted ascending by timestamp with tz-aware timestamp strings
if 'timestamp' in error_logs_df.columns and error_total_rows>0:
    err_top5 = error_logs_df.sort_values('timestamp', ascending=True).head(5).copy()
    err_top5['timestamp'] = pd.to_datetime(err_top5['timestamp'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S%z')
    error_top5 = err_top5[['timestamp','cmdb_id','message']].reset_index(drop=True)
else:
    error_top5 = error_logs_df.head(0)[['timestamp','cmdb_id','message']]

# Display compact summaries (variables). Keep outputs compact as requested.
metric_unique_cmdbs, metric_unique_kpis, metric_total_rows, metric_distinct_pairs, metric_summary_top, trace_unique_cmdbs, trace_unique_kpis, trace_total_rows, trace_distinct_pairs, trace_summary_top, log_unique_cmdbs, log_unique_kpis, log_total_rows, log_distinct_pairs, log_summary_top, error_total_rows, error_unique_cmdbs, error_top5

```
Out[1]:
```
```python
# Summarize the previously computed results in plain English as a single string.
# Uses kernel variables produced earlier: metric_unique_cmdbs, metric_unique_kpis, metric_total_rows, metric_distinct_pairs, metric_summary_top,
# trace_unique_cmdbs, trace_unique_kpis, trace_total_rows, trace_distinct_pairs, trace_summary_top,
# log_unique_cmdbs, log_unique_kpis, log_total_rows, log_distinct_pairs, log_summary_top,
# error_total_rows, error_unique_cmdbs, error_top5

def format_top(df, n=3):
    # format first n rows of a summary dataframe into compact CSV-like lines
    lines = []
    for _, r in df.head(n).iterrows():
        lines.append(f"{r['cmdb_id']},{r['KPI_name']},{int(r['row_count'])},{r['min_timestamp']},{r['max_timestamp']}")
    return "\n".join(lines)

summary = (
    "Summary of loaded telemetry files:\n\n"
    "metric.csv:\n"
    f"- Total rows: {metric_total_rows}\n"
    f"- Unique cmdb_id count: {len(metric_unique_cmdbs)}; values: {metric_unique_cmdbs}\n"
    f"- Unique KPI names count: {len(metric_unique_kpis)}\n"
    f"- Distinct (cmdb_id, KPI) pairs: {metric_distinct_pairs}\n"
    "- Top 3 (cmdb_id,KPI,row_count,min_timestamp,max_timestamp):\n"
    f"{format_top(metric_summary_top, n=3)}\n\n"
    "trace.csv:\n"
    f"- Total rows: {trace_total_rows}\n"
    f"- Unique cmdb_id count: {len(trace_unique_cmdbs)}; values: {trace_unique_cmdbs}\n"
    f"- Unique trace names count: {len(trace_unique_kpis)}\n"
    f"- Distinct (cmdb_id, trace) pairs: {trace_distinct_pairs}\n"
    "- Top 3 (cmdb_id,KPI,row_count,min_timestamp,max_timestamp):\n"
    f"{format_top(trace_summary_top, n=3)}\n\n"
    "log.csv:\n"
    f"- Total rows: {log_total_rows}\n"
    f"- Unique cmdb_id count: {len(log_unique_cmdbs)}; values: {log_unique_cmdbs}\n"
    f"- Unique log names count: {len(log_unique_kpis)}\n"
    f"- Distinct (cmdb_id, log) pairs: {log_distinct_pairs}\n"
    "- Top 3 (cmdb_id,KPI,row_count,min_timestamp,max_timestamp):\n"
    f"{format_top(log_summary_top, n=3)}\n\n"
    "error_logs.csv:\n"
    f"- Total rows: {error_total_rows}\n"
    f"- Unique cmdb_id values: {error_unique_cmdbs}\n"
    f"- Top 5 rows by timestamp (timestamp,cmdb_id,message):\n"
    f"{'' if error_top5.shape[0]==0 else chr(10).join(error_top5.astype(str).apply(lambda r: ','.join(r.values), axis=1).tolist())}\n\n"
    "Notes:\n"
    "- metric.csv is the largest file (28,453 rows) with many KPIs (count shown above).\n"
    "- trace and log files cover fewer distinct KPI pairs; trace top pairs mostly have 30 rows each for IG01/IG02.\n"
    "- error_logs.csv contains no rows in this dataset.\n"
    "End of summary."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(['IG01', 'IG02', 'MG01', 'MG02', 'Mysql01', 'Mysql02', 'Redis01', 'Redis02', 'ServiceTest1', 'ServiceTest10', 'ServiceTest11', 'ServiceTest2', 'ServiceTest3', 'ServiceTest4', 'ServiceTest5', 'ServiceTest6', 'ServiceTest7', 'ServiceTest8', 'ServiceTest9', 'Tomcat01', 'Tomcat02', 'Tomcat03', 'Tomcat04', 'apache01', 'apache02', 'dockerA1', 'dockerA2', 'dockerB1', 'dockerB2'], ['Container-DOCKER_CONTAINER_2d16f5b2e830--bcou-trace-st-uat-statefulset-1--bcou--UATWKR18_CpuPercent', 'Container-DOCKER_CONTAINER_2d16f5b2e830--bcou-trace-st-uat-statefulset-1--bcou--UATWKR18_MemLimit', 'Container-DOCKER_CONTAINER_2d16f5b2e830--bcou-trace-st-uat-statefulset-1--bcou--UATWKR18_MemPercent', 'Container-DOCKER_CONTAINER_2d16f5b2e830--bcou-trace-st-uat-statefulset-1--bcou--UATWKR18_MemUsage', 'Container-DOCKER_CONTAINER_2d16f5b2e830--bcou-trace-st-uat-statefulset-1--bcou--UATWKR18_NetworkTxBytes', 'Container-DOCKER_CONTAINER_94eca4f96efe--bcou-role-st-uat-statefulset-1--bcou--UATWKR04_CpuPercent', 'Container-DOCKER_CONTAINER_94eca4f96efe--bcou-role-st-uat-statefulset-1--bcou--UATWKR04_MemLimit', 'Container-DOCKER_CONTAINER_94eca4f96efe--bcou-role-st-uat-statefulset-1--bcou--UATWKR04_MemPercent', 'Container-DOCKER_CONTAINER_94eca4f96efe--bcou-role-st-uat-statefulset-1--bcou--UATWKR04_MemUsage', 'Container-DOCKER_CONTAINER_94eca4f96efe--bcou-role-st-uat-statefulset-1--bcou--UATWKR04_NetworkRxBytes', 'Container-DOCKER_CONTAINER_94eca4f96efe--bcou-role-st-uat-statefulset-1--bcou--UATWKR04_NetworkTxBytes', 'Container-DOCKER_CONTAINER_b30097144a13--bcou-trace-st-uat-statefulset-0--bcou--UATWKR04_CpuPercent', 'Container-DOCKER_CONTAINER_b30097144a13--bcou-trace-st-uat-statefulset-0--bcou--UATWKR04_MemLimit', 'Container-DOCKER_CONTAINER_b30097144a13--bcou-trace-st-uat-statefulset-0--bcou--UATWKR04_MemPercent', 'Container-DOCKER_CONTAINER_b30097144a13--bcou-trace-st-uat-statefulset-0--bcou--UATWKR04_MemUsage', 'Container-DOCKER_CONTAINER_b30097144a13--bcou-trace-st-uat-statefulset-0--bcou--UATWKR04_NetworkRxBytes', 'Container-DOCKER_CONTAINER_b30097144a13--bcou-trace-st-uat-statefulset-0--bcou--UATWKR04_NetworkTxBytes', 'Container-DOCKER_CONTAINER_b6760337dc49--bcou-role-st-uat-statefulset-0--bcou--UATWKR18_CpuPercent', 'Container-DOCKER_CONTAINER_b6760337dc49--bcou-role-st-uat-statefulset-0--bcou--UATWKR18_MemLimit', 'Container-DOCKER_CONTAINER_b6760337dc49--bcou-role-st-uat-statefulset-0--bcou--UATWKR18_MemPercent', 'Container-DOCKER_CONTAINER_b6760337dc49--bcou-role-st-uat-statefulset-0--bcou--UATWKR18_MemUsage', 'Container-DOCKER_CONTAINER_b6760337dc49--bcou-role-st-uat-statefulset-0--bcou--UATWKR18_NetworkRxBytes', 'Container-DOCKER_CONTAINER_b6760337dc49--bcou-role-st-uat-statefulset-0--bcou--UATWKR18_NetworkTxBytes', 'Container-DOCKER_CONTAINER_c67422614c81--bcou-role-st-uat-statefulset-1--bcou--UATWKR02_CpuPercent', 'Container-DOCKER_CONTAINER_c67422614c81--bcou-role-st-uat-statefulset-1--bcou--UATWKR02_MemLimit', 'Container-DOCKER_CONTAINER_c67422614c81--bcou-role-st-uat-statefulset-1--bcou--UATWKR02_MemPercent', 'Container-DOCKER_CONTAINER_c67422614c81--bcou-role-st-uat-statefulset-1--bcou--UATWKR02_MemUsage', 'Container-DOCKER_CONTAINER_c67422614c81--bcou-role-st-uat-statefulset-1--bcou--UATWKR02_NetworkTxBytes', 'JVM-Memory_7778_JVM_Memory_HeapMemoryMax', 'JVM-Memory_7778_JVM_Memory_HeapMemoryUsage', 'JVM-Memory_7778_JVM_Memory_HeapMemoryUsed', 'JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed', 'JVM-Memory_7779_JVM_Memory_HeapMemoryMax', 'JVM-Memory_7779_JVM_Memory_HeapMemoryUsage', 'JVM-Memory_7779_JVM_Memory_HeapMemoryUsed', 'JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed', 'JVM-Operating System_7778_JVM_JVM_CPULoad', 'JVM-Operating System_7779_JVM_JVM_CPULoad', 'JVM-Runtime_7778_JVM_JVM_Uptime', 'JVM-Runtime_7779_JVM_JVM_Uptime', 'JVM-Threads_7778_JVM_ThreadCount_Threads', 'JVM-Threads_7779_JVM_ThreadCount_Threads', 'Mysql-MySQL_3306_Aborted Clients', 'Mysql-MySQL_3306_Aborted Connects', 'Mysql-MySQL_3306_Binlog Cache Disk Use', 'Mysql-MySQL_3306_Binlog Cache Use', 'Mysql-MySQL_3306_Binlog stmt cache disk use', 'Mysql-MySQL_3306_Binlog stmt cache use', 'Mysql-MySQL_3306_Bytes Received', 'Mysql-MySQL_3306_Bytes Sent', 'Mysql-MySQL_3306_Com Delete', 'Mysql-MySQL_3306_Com Delete Multi', 'Mysql-MySQL_3306_Com Insert', 'Mysql-MySQL_3306_Com Insert Select', 'Mysql-MySQL_3306_Com Load', 'Mysql-MySQL_3306_Com Replace', 'Mysql-MySQL_3306_Com Replace Select', 'Mysql-MySQL_3306_Com Select', 'Mysql-MySQL_3306_Com Update', 'Mysql-MySQL_3306_Com Update Multi', 'Mysql-MySQL_3306_Connections', 'Mysql-MySQL_3306_Created Tmp Disk Tables', 'Mysql-MySQL_3306_Created Tmp Files', 'Mysql-MySQL_3306_Created Tmp Tables', 'Mysql-MySQL_3306_CurrentSQLMaxRunningTime', 'Mysql-MySQL_3306_GetConnectedStateOfMysqld', 'Mysql-MySQL_3306_GetResponseTimeOfMysqld', 'Mysql-MySQL_3306_Handler Commit', 'Mysql-MySQL_3306_Handler Delete', 'Mysql-MySQL_3306_Handler Read First', 'Mysql-MySQL_3306_Handler Read Key', 'Mysql-MySQL_3306_Handler Read Next', 'Mysql-MySQL_3306_Handler Read Prev', 'Mysql-MySQL_3306_Handler Read Rnd', 'Mysql-MySQL_3306_Handler Read Rnd Next', 'Mysql-MySQL_3306_Handler Rollback', 'Mysql-MySQL_3306_Handler Savepoint', 'Mysql-MySQL_3306_Handler Savepoint Rollback', 'Mysql-MySQL_3306_Handler Update', 'Mysql-MySQL_3306_Handler Write', 'Mysql-MySQL_3306_Innodb Row Lock Time', 'Mysql-MySQL_3306_Innodb Row Lock Waits', 'Mysql-MySQL_3306_Innodb buffer pool pages dirty', 'Mysql-MySQL_3306_Innodb buffer pool pages flushed', 'Mysql-MySQL_3306_Innodb buffer pool pages free', 'Mysql-MySQL_3306_Innodb buffer pool pages misc', 'Mysql-MySQL_3306_Innodb buffer pool pages total', 'Mysql-MySQL_3306_Innodb buffer pool read ahead', 'Mysql-MySQL_3306_Innodb buffer pool read ahead evicted', 'Mysql-MySQL_3306_Innodb buffer pool read ahead rnd', 'Mysql-MySQL_3306_Innodb buffer pool read requests', 'Mysql-MySQL_3306_Innodb buffer pool reads', 'Mysql-MySQL_3306_Innodb buffer pool wait free', 'Mysql-MySQL_3306_Innodb buffer pool write requests', 'Mysql-MySQL_3306_Innodb data fsyncs', 'Mysql-MySQL_3306_Innodb data pending fsyncs', 'Mysql-MySQL_3306_Innodb data pending reads', 'Mysql-MySQL_3306_Innodb data pending writes', 'Mysql-MySQL_3306_Innodb data read', 'Mysql-MySQL_3306_Innodb data reads', 'Mysql-MySQL_3306_Innodb data writes', 'Mysql-MySQL_3306_Innodb data written', 'Mysql-MySQL_3306_Innodb dblwr pages written', 'Mysql-MySQL_3306_Innodb dblwr writes', 'Mysql-MySQL_3306_Innodb log waits', 'Mysql-MySQL_3306_Innodb log write requests', 'Mysql-MySQL_3306_Innodb log writes', 'Mysql-MySQL_3306_Innodb open files  num', 'Mysql-MySQL_3306_Innodb os log fsyncs', 'Mysql-MySQL_3306_Innodb os log pending fsyncs', 'Mysql-MySQL_3306_Innodb os log pending writes', 'Mysql-MySQL_3306_Innodb os log written', 'Mysql-MySQL_3306_Innodb pages created', 'Mysql-MySQL_3306_Innodb pages read', 'Mysql-MySQL_3306_Innodb pages written', 'Mysql-MySQL_3306_Innodb row lock current waits', 'Mysql-MySQL_3306_Innodb row lock time avg', 'Mysql-MySQL_3306_Innodb row lock time max', 'Mysql-MySQL_3306_Key Read Requests', 'Mysql-MySQL_3306_Key Reads', 'Mysql-MySQL_3306_Key Write Requests', 'Mysql-MySQL_3306_Key Writes', 'Mysql-MySQL_3306_LongestTrxActiveTime', 'Mysql-MySQL_3306_Max Used Connections', 'Mysql-MySQL_3306_Max trx rows locked', 'Mysql-MySQL_3306_MaxConnections', 'Mysql-MySQL_3306_MaxTrxRowsModified', 'Mysql-MySQL_3306_MySQL  Queries', 'Mysql-MySQL_3306_Open Files', 'Mysql-MySQL_3306_Open Tables', 'Mysql-MySQL_3306_Opened Tables', 'Mysql-MySQL_3306_Opened table definitions', 'Mysql-MySQL_3306_Qcache Free Blocks', 'Mysql-MySQL_3306_Qcache Free Memory', 'Mysql-MySQL_3306_Qcache Hits', 'Mysql-MySQL_3306_Qcache Inserts', 'Mysql-MySQL_3306_Qcache Lowmem Prunes', 'Mysql-MySQL_3306_Qcache Not Cached', 'Mysql-MySQL_3306_Qcache Queries In Cache', 'Mysql-MySQL_3306_Qcache Total Blocks', 'Mysql-MySQL_3306_Questions', 'Mysql-MySQL_3306_Rows Read', 'Mysql-MySQL_3306_Select Full Join', 'Mysql-MySQL_3306_Select Full Range Join', 'Mysql-MySQL_3306_Select Range', 'Mysql-MySQL_3306_Select Range Check', 'Mysql-MySQL_3306_Select Scan', 'Mysql-MySQL_3306_Slave Open Temp Tables', 'Mysql-MySQL_3306_Slow launch threads', 'Mysql-MySQL_3306_SlowQueries', 'Mysql-MySQL_3306_Sort Merge Passes', 'Mysql-MySQL_3306_Sort Range', 'Mysql-MySQL_3306_Sort Rows', 'Mysql-MySQL_3306_Sort Scan', 'Mysql-MySQL_3306_Table Locks Immediate', 'Mysql-MySQL_3306_Table Locks Waited', 'Mysql-MySQL_3306_Table open cache hits', 'Mysql-MySQL_3306_Table open cache misses', 'Mysql-MySQL_3306_Table open cache overflows', 'Mysql-MySQL_3306_Tc log max pages used', 'Mysql-MySQL_3306_Tc log page waits', 'Mysql-MySQL_3306_Threads Cached', 'Mysql-MySQL_3306_Threads Created', 'Mysql-MySQL_3306_ThreadsConnected', 'Mysql-MySQL_3306_ThreadsRunning', 'Mysql-MySQL_3306_max trx lock memory bytes', 'OSLinux-CPU_CPU-0_SingleCpuUtil', 'OSLinux-CPU_CPU-0_SingleCpuidle', 'OSLinux-CPU_CPU-1_SingleCpuUtil', 'OSLinux-CPU_CPU-1_SingleCpuidle', 'OSLinux-CPU_CPU-2_SingleCpuUtil', 'OSLinux-CPU_CPU-2_SingleCpuidle', 'OSLinux-CPU_CPU-3_SingleCpuUtil', 'OSLinux-CPU_CPU-3_SingleCpuidle', 'OSLinux-CPU_CPU_CPUCpuUtil', 'OSLinux-CPU_CPU_CPULoad', 'OSLinux-CPU_CPU_CPUSysTime', 'OSLinux-CPU_CPU_CPUUserTime', 'OSLinux-CPU_CPU_CPUWio', 'OSLinux-CPU_CPU_CPUidleutil', 'OSLinux-NTP_197.30.1.67_NtpServerTimeOffset', 'OSLinux-NTP_197.30.1.68_NtpServerTimeOffset', 'OSLinux-OSLinux_FILESYSTEM_-_FSAvailableSpace', 'OSLinux-OSLinux_FILESYSTEM_-_FSCapacity', 'OSLinux-OSLinux_FILESYSTEM_-_FSInodeUsedPercent', 'OSLinux-OSLinux_FILESYSTEM_-_FSUsedSpace', 'OSLinux-OSLinux_FILESYSTEM_-apache_FSAvailableSpace', 'OSLinux-OSLinux_FILESYSTEM_-apache_FSCapacity', 'OSLinux-OSLinux_FILESYSTEM_-apache_FSInodeUsedPercent', 'OSLinux-OSLinux_FILESYSTEM_-apache_FSUsedSpace', 'OSLinux-OSLinux_FILESYSTEM_-app_FSAvailableSpace', 'OSLinux-OSLinux_FILESYSTEM_-app_FSCapacity', 'OSLinux-OSLinux_FILESYSTEM_-app_FSInodeUsedPercent', 'OSLinux-OSLinux_FILESYSTEM_-app_FSUsedSpace', 'OSLinux-OSLinux_FILESYSTEM_-boot_FSAvailableSpace', 'OSLinux-OSLinux_FILESYSTEM_-boot_FSCapacity', 'OSLinux-OSLinux_FILESYSTEM_-boot_FSInodeUsedPercent', 'OSLinux-OSLinux_FILESYSTEM_-boot_FSUsedSpace', 'OSLinux-OSLinux_FILESYSTEM_-cmbc_admin_FSAvailableSpace', 'OSLinux-OSLinux_FILESYSTEM_-cmbc_admin_FSCapacity', 'OSLinux-OSLinux_FILESYSTEM_-cmbc_admin_FSInodeUsedPercent', 'OSLinux-OSLinux_FILESYSTEM_-cmbc_admin_FSUsedSpace', 'OSLinux-OSLinux_FILESYSTEM_-home_FSAvailableSpace', 'OSLinux-OSLinux_FILESYSTEM_-home_FSCapacity', 'OSLinux-OSLinux_FILESYSTEM_-home_FSInodeUsedPercent', 'OSLinux-OSLinux_FILESYSTEM_-home_FSUsedSpace', 'OSLinux-OSLinux_FILESYSTEM_-mysql_FSAvailableSpace', 'OSLinux-OSLinux_FILESYSTEM_-mysql_FSCapacity', 'OSLinux-OSLinux_FILESYSTEM_-mysql_FSInodeUsedPercent', 'OSLinux-OSLinux_FILESYSTEM_-mysql_FSUsedSpace', 'OSLinux-OSLinux_FILESYSTEM_-mysqlbak_FSAvailableSpace', 'OSLinux-OSLinux_FILESYSTEM_-mysqlbak_FSCapacity', 'OSLinux-OSLinux_FILESYSTEM_-mysqlbak_FSInodeUsedPercent', 'OSLinux-OSLinux_FILESYSTEM_-mysqlbak_FSUsedSpace', 'OSLinux-OSLinux_FILESYSTEM_-mysqldata_FSAvailableSpace', 'OSLinux-OSLinux_FILESYSTEM_-mysqldata_FSCapacity', 'OSLinux-OSLinux_FILESYSTEM_-mysqldata_FSInodeUsedPercent', 'OSLinux-OSLinux_FILESYSTEM_-mysqldata_FSUsedSpace', 'OSLinux-OSLinux_FILESYSTEM_-tmp_FSAvailableSpace', 'OSLinux-OSLinux_FILESYSTEM_-tmp_FSCapacity', 'OSLinux-OSLinux_FILESYSTEM_-tmp_FSInodeUsedPercent', 'OSLinux-OSLinux_FILESYSTEM_-tmp_FSUsedSpace', 'OSLinux-OSLinux_FILESYSTEM_-tomcat_FSAvailableSpace', 'OSLinux-OSLinux_FILESYSTEM_-tomcat_FSCapacity', 'OSLinux-OSLinux_FILESYSTEM_-tomcat_FSInodeUsedPercent', 'OSLinux-OSLinux_FILESYSTEM_-tomcat_FSUsedSpace', 'OSLinux-OSLinux_FILESYSTEM_-usr-openv_FSAvailableSpace', 'OSLinux-OSLinux_FILESYSTEM_-usr-openv_FSCapacity', 'OSLinux-OSLinux_FILESYSTEM_-usr-openv_FSInodeUsedPercent', 'OSLinux-OSLinux_FILESYSTEM_-usr-openv_FSUsedSpace', 'OSLinux-OSLinux_FILE_-home-zabbix_DirSizeMB', 'OSLinux-OSLinux_FILE_-tmp-zabbix_agentd.log_FileSizeMB', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKAvgServ', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPercentBusy', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKReadWrite', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWTps', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWrite', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKAvgServ', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPercentBusy', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKReadWrite', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWTps', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWrite', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKAvgServ', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKBps', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKPercentBusy', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKRTps', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKRead', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKReadWrite', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKTps', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKWTps', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKWrite', 'OSLinux-OSLinux_MEMORY_MEMORY_CacheMem', 'OSLinux-OSLinux_MEMORY_MEMORY_MEMFreeMem', 'OSLinux-OSLinux_MEMORY_MEMORY_MEMTotalMem', 'OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc', 'OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc', 'OSLinux-OSLinux_MEMORY_MEMORY_UserMem', 'OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT', 'OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT', 'OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNum', 'OSLinux-OSLinux_NETWORK_ens160_NETBandwidthUtil', 'OSLinux-OSLinux_NETWORK_ens160_NETInErr', 'OSLinux-OSLinux_NETWORK_ens160_NETInErrPrc', 'OSLinux-OSLinux_NETWORK_ens160_NETKBTotalPerSec', 'OSLinux-OSLinux_NETWORK_ens160_NETOutErr', 'OSLinux-OSLinux_NETWORK_ens160_NETOutErrPrcc', 'OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn', 'OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut', 'OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies', 'OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMem', 'OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMemPerc', 'OSLinux-OSLinux_PROCESS_apache_10001_PROCPPCount', 'OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_PROCPPCPUPerc', 'OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount', 'OSLinux-OSLinux_SWAP_SWAP_SWPTotSwapSize', 'OSLinux-OSLinux_SWAP_SWAP_SWPTotSwapUsedPercent', 'OSLinux-OSLinux_SWAP_SWAP_Si', 'OSLinux-OSLinux_SWAP_SWAP_So', 'OSLinux-OSLinux_SYSTEM_SYSTEM_Check-DefaultRoute', 'OSLinux-OSLinux_SYSTEM_SYSTEM_Check-Hostname', 'OSLinux-OSLinux_ZABBIX_Host_Uptime', 'Tomcat-MEMORY_7441-MEMORY_JVMFreeMemory', 'Tomcat-MEMORY_7441-MEMORY_JVMMaxMemory', 'Tomcat-MEMORY_7441-MEMORY_JVMMemoryUsedPercent', 'Tomcat-MEMORY_7441-MEMORY_JVMTotalMemory', 'Tomcat-MEMORY_7441-MEMORY_JVMUsedMemory', 'Tomcat-Requests_7441-"http-nio-8003"_ErrorCountRequestInfo', 'Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRequestInfo', 'Tomcat-Requests_7441-"http-nio-8003"_ProcessingTimeRequestInfo', 'Tomcat-Requests_7441-"http-nio-8003"_RequestCountRequestInfo', 'Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter', 'Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCounter', 'Tomcat-Sessions_7441--UOCP_SESSIONRejectedSessions', 'Tomcat-Sessions_7441--_SESSIONActiveCounter', 'Tomcat-Sessions_7441--_SESSIONKeepaliveCounter', 'Tomcat-Sessions_7441--_SESSIONRejectedSessions', 'Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SESSIONActiveCounter', 'Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SESSIONKeepaliveCounter', 'Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SESSIONRejectedSessions', 'Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SESSIONActiveCounter', 'Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SESSIONKeepaliveCounter', 'Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SESSIONRejectedSessions', 'Tomcat-Threads_7441-"http-nio-8003"_CurrentThreadCountThreadInfo', 'Tomcat-Threads_7441-"http-nio-8003"_CurrentThreadsBusyThreadInfo', 'Tomcat-Threads_7441-"http-nio-8003"_MaxThreadsThreadInfo', 'app.cnt', 'app.mrt', 'app.rr', 'app.sr', 'redis-Redis_6379_Redis  (aof_enabled)', 'redis-Redis_6379_Redis  (aof_rewrite_in_progress)', 'redis-Redis_6379_Redis  (aof_rewrite_scheduled)', 'redis-Redis_6379_Redis  (blocked_clients)', 'redis-Redis_6379_Redis  (client_biggest_input_buf)', 'redis-Redis_6379_Redis  (client_longest_output_list)', 'redis-Redis_6379_Redis  (connected_clients)', 'redis-Redis_6379_Redis  (connected_slaves)', 'redis-Redis_6379_Redis  (evicted_keys)', 'redis-Redis_6379_Redis  (expired_keys)', 'redis-Redis_6379_Redis  (instantaneous_ops_per_sec)', 'redis-Redis_6379_Redis  (keyspace_hits)', 'redis-Redis_6379_Redis  (keyspace_misses)', 'redis-Redis_6379_Redis  (latest_fork_usec)', 'redis-Redis_6379_Redis  (loading)', 'redis-Redis_6379_Redis  (lru_clock)', 'redis-Redis_6379_Redis  (maxmemory)', 'redis-Redis_6379_Redis  (mem_fragmentation_ratio)', 'redis-Redis_6379_Redis  (pubsub_channels)', 'redis-Redis_6379_Redis  (rdb_bgsave_in_progress)', 'redis-Redis_6379_Redis  (rdb_changes_since_last_save)', 'redis-Redis_6379_Redis  (redis_git_dirty)', 'redis-Redis_6379_Redis  (rejected_connections)', 'redis-Redis_6379_Redis  (total_commands_processed)', 'redis-Redis_6379_Redis  (total_connections_received)', 'redis-Redis_6379_Redis  (uptime_in_seconds)', 'redis-Redis_6379_Redis  (used_cpu_sys)', 'redis-Redis_6379_Redis  (used_cpu_sys_children)', 'redis-Redis_6379_Redis  (used_cpu_user)', 'redis-Redis_6379_Redis  (used_cpu_user_children)', 'redis-Redis_6379_Redis  (used_memory)', 'redis-Redis_6379_Redis  (used_memory_peak)', 'redis-Redis_6379_Redis  (used_memory_rss)', 'redis-Redis_6379_redis server'], 28453, 1708,      cmdb_id                                           KPI_name  row_count             min_timestamp             max_timestamp
0   Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...         31  2021-03-07 00:30:00+0800  2021-03-07 01:00:00+0800
1       IG01           JVM-Threads_7778_JVM_ThreadCount_Threads         31  2021-03-07 00:30:00+0800  2021-03-07 01:00:00+0800
2   Tomcat04     Tomcat-Sessions_7441--_SESSIONRejectedSessions         31  2021-03-07 00:30:00+0800  2021-03-07 01:00:00+0800
3   Tomcat04     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter         31  2021-03-07 00:30:00+0800  2021-03-07 01:00:00+0800
4   Tomcat04        Tomcat-Sessions_7441--_SESSIONActiveCounter         31  2021-03-07 00:30:00+0800  2021-03-07 01:00:00+0800
5   Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...         31  2021-03-07 00:30:00+0800  2021-03-07 01:00:00+0800
6   Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...         31  2021-03-07 00:30:00+0800  2021-03-07 01:00:00+0800
7   apache02                         OSLinux-CPU_CPU_CPUSysTime         31  2021-03-07 00:30:00+0800  2021-03-07 01:00:00+0800
8   apache02                        OSLinux-CPU_CPU_CPUUserTime         31  2021-03-07 00:30:00+0800  2021-03-07 01:00:00+0800
9   apache02                             OSLinux-CPU_CPU_CPUWio         31  2021-03-07 00:30:00+0800  2021-03-07 01:00:00+0800
10  apache02                        OSLinux-CPU_CPU_CPUidleutil         31  2021-03-07 00:30:00+0800  2021-03-07 01:00:00+0800
11  apache02                            OSLinux-CPU_CPU_CPULoad         31  2021-03-07 00:30:00+0800  2021-03-07 01:00:00+0800
12  apache01                 OSLinux-OSLinux_ZABBIX_Host_Uptime         31  2021-03-07 00:30:00+0800  2021-03-07 01:00:00+0800
13  apache01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies         31  2021-03-07 00:30:00+0800  2021-03-07 01:00:00+0800
14  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...         31  2021-03-07 00:30:00+0800  2021-03-07 01:00:00+0800
15  apache01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...         31  2021-03-07 00:30:00+0800  2021-03-07 01:00:00+0800
16  apache01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount         31  2021-03-07 00:30:00+0800  2021-03-07 01:00:00+0800
17  Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...         31  2021-03-07 00:30:00+0800  2021-03-07 01:00:00+0800
18   Mysql01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies         31  2021-03-07 00:30:00+0800  2021-03-07 01:00:00+0800
19   Mysql01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...         31  2021-03-07 00:30:00+0800  2021-03-07 01:00:00+0800, ['IG01', 'IG02', 'MG01', 'MG02', 'Tomcat01', 'Tomcat02', 'Tomcat03', 'Tomcat04', 'dockerA1', 'dockerA2', 'dockerB1', 'dockerB2'], ['trace.from_IG01.duration_mean', 'trace.from_IG01.duration_p95', 'trace.from_IG01.row_count', 'trace.from_IG02.duration_mean', 'trace.from_IG02.duration_p95', 'trace.from_IG02.row_count', 'trace.from_MG01.duration_mean', 'trace.from_MG01.duration_p95', 'trace.from_MG01.row_count', 'trace.from_MG02.duration_mean', 'trace.from_MG02.duration_p95', 'trace.from_MG02.row_count', 'trace.from_Tomcat01.duration_mean', 'trace.from_Tomcat01.duration_p95', 'trace.from_Tomcat01.row_count', 'trace.from_Tomcat02.duration_mean', 'trace.from_Tomcat02.duration_p95', 'trace.from_Tomcat02.row_count', 'trace.from_Tomcat03.duration_mean', 'trace.from_Tomcat03.duration_p95', 'trace.from_Tomcat03.row_count', 'trace.from_Tomcat04.duration_mean', 'trace.from_Tomcat04.duration_p95', 'trace.from_Tomcat04.row_count', 'trace.from_dockerA1.duration_mean', 'trace.from_dockerA1.duration_p95', 'trace.from_dockerA1.row_count', 'trace.from_dockerA2.duration_mean', 'trace.from_dockerA2.duration_p95', 'trace.from_dockerA2.row_count', 'trace.from_dockerB1.duration_mean', 'trace.from_dockerB1.duration_p95', 'trace.from_dockerB1.row_count', 'trace.from_dockerB2.duration_mean', 'trace.from_dockerB2.duration_p95', 'trace.from_dockerB2.row_count', 'trace.self.duration_mean', 'trace.self.duration_p95', 'trace.self.row_count', 'trace.to_MG01.duration_mean', 'trace.to_MG01.duration_p95', 'trace.to_MG01.row_count', 'trace.to_MG02.duration_mean', 'trace.to_MG02.duration_p95', 'trace.to_MG02.row_count', 'trace.to_Tomcat01.duration_mean', 'trace.to_Tomcat01.duration_p95', 'trace.to_Tomcat01.row_count', 'trace.to_Tomcat02.duration_mean', 'trace.to_Tomcat02.duration_p95', 'trace.to_Tomcat02.row_count', 'trace.to_Tomcat03.duration_mean', 'trace.to_Tomcat03.duration_p95', 'trace.to_Tomcat03.row_count', 'trace.to_Tomcat04.duration_mean', 'trace.to_Tomcat04.duration_p95', 'trace.to_Tomcat04.row_count', 'trace.to_dockerA1.duration_mean', 'trace.to_dockerA1.duration_p95', 'trace.to_dockerA1.row_count', 'trace.to_dockerA2.duration_mean', 'trace.to_dockerA2.duration_p95', 'trace.to_dockerA2.row_count', 'trace.to_dockerB1.duration_mean', 'trace.to_dockerB1.duration_p95', 'trace.to_dockerB1.row_count', 'trace.to_dockerB2.duration_mean', 'trace.to_dockerB2.duration_p95', 'trace.to_dockerB2.row_count'], 6498, 228,    cmdb_id                         KPI_name  row_count             min_timestamp             max_timestamp
0     IG01         trace.self.duration_mean         30  2021-03-07 00:30:00+0800  2021-03-07 00:59:00+0800
1     IG01          trace.self.duration_p95         30  2021-03-07 00:30:00+0800  2021-03-07 00:59:00+0800
2     IG01             trace.self.row_count         30  2021-03-07 00:30:00+0800  2021-03-07 00:59:00+0800
3     IG01  trace.to_Tomcat01.duration_mean         30  2021-03-07 00:30:00+0800  2021-03-07 00:59:00+0800
4     IG01   trace.to_Tomcat01.duration_p95         30  2021-03-07 00:30:00+0800  2021-03-07 00:59:00+0800
5     IG01      trace.to_Tomcat01.row_count         30  2021-03-07 00:30:00+0800  2021-03-07 00:59:00+0800
6     IG01  trace.to_Tomcat02.duration_mean         30  2021-03-07 00:30:00+0800  2021-03-07 00:59:00+0800
7     IG01   trace.to_Tomcat02.duration_p95         30  2021-03-07 00:30:00+0800  2021-03-07 00:59:00+0800
8     IG01      trace.to_Tomcat02.row_count         30  2021-03-07 00:30:00+0800  2021-03-07 00:59:00+0800
9     IG01  trace.to_Tomcat03.duration_mean         30  2021-03-07 00:30:00+0800  2021-03-07 00:59:00+0800
10    IG01   trace.to_Tomcat03.duration_p95         30  2021-03-07 00:30:00+0800  2021-03-07 00:59:00+0800
11    IG01      trace.to_Tomcat03.row_count         30  2021-03-07 00:30:00+0800  2021-03-07 00:59:00+0800
12    IG01  trace.to_Tomcat04.duration_mean         30  2021-03-07 00:30:00+0800  2021-03-07 00:59:00+0800
13    IG01   trace.to_Tomcat04.duration_p95         30  2021-03-07 00:30:00+0800  2021-03-07 00:59:00+0800
14    IG01      trace.to_Tomcat04.row_count         30  2021-03-07 00:30:00+0800  2021-03-07 00:59:00+0800
15    IG02         trace.self.duration_mean         30  2021-03-07 00:30:00+0800  2021-03-07 00:59:00+0800
16    IG02          trace.self.duration_p95         30  2021-03-07 00:30:00+0800  2021-03-07 00:59:00+0800
17    IG02             trace.self.row_count         30  2021-03-07 00:30:00+0800  2021-03-07 00:59:00+0800
18    IG02  trace.to_Tomcat01.duration_mean         30  2021-03-07 00:30:00+0800  2021-03-07 00:59:00+0800
19    IG02   trace.to_Tomcat01.duration_p95         30  2021-03-07 00:30:00+0800  2021-03-07 00:59:00+0800, ['Mysql01', 'Tomcat01', 'Tomcat02', 'Tomcat03', 'Tomcat04', 'apache01', 'apache02'], ['log.error_count', 'log.gc_pause_max', 'log.row_count'], 394, 18,      cmdb_id          KPI_name  row_count             min_timestamp             max_timestamp
0   Tomcat02     log.row_count         31  2021-03-07 00:30:00+0800  2021-03-07 01:00:00+0800
1   Tomcat02   log.error_count         31  2021-03-07 00:30:00+0800  2021-03-07 01:00:00+0800
2   apache02     log.row_count         31  2021-03-07 00:30:00+0800  2021-03-07 01:00:00+0800
3   apache01     log.row_count         31  2021-03-07 00:30:00+0800  2021-03-07 01:00:00+0800
4   Tomcat03   log.error_count         31  2021-03-07 00:30:00+0800  2021-03-07 01:00:00+0800
5   apache02   log.error_count         31  2021-03-07 00:30:00+0800  2021-03-07 01:00:00+0800
6   apache01   log.error_count         31  2021-03-07 00:30:00+0800  2021-03-07 01:00:00+0800
7   Tomcat04     log.row_count         31  2021-03-07 00:30:00+0800  2021-03-07 01:00:00+0800
8   Tomcat04   log.error_count         31  2021-03-07 00:30:00+0800  2021-03-07 01:00:00+0800
9   Tomcat03     log.row_count         31  2021-03-07 00:30:00+0800  2021-03-07 01:00:00+0800
10  Tomcat01   log.error_count         30  2021-03-07 00:30:00+0800  2021-03-07 00:59:00+0800
11  Tomcat01     log.row_count         30  2021-03-07 00:30:00+0800  2021-03-07 00:59:00+0800
12  Tomcat02  log.gc_pause_max          8  2021-03-07 00:31:00+0800  2021-03-07 00:55:00+0800
13  Tomcat03  log.gc_pause_max          5  2021-03-07 00:31:00+0800  2021-03-07 00:58:00+0800
14  Tomcat04  log.gc_pause_max          4  2021-03-07 00:34:00+0800  2021-03-07 00:56:00+0800
15  Tomcat01  log.gc_pause_max          3  2021-03-07 00:35:00+0800  2021-03-07 00:53:00+0800
16   Mysql01     log.row_count          2  2021-03-07 00:31:00+0800  2021-03-07 00:55:00+0800
17   Mysql01   log.error_count          2  2021-03-07 00:31:00+0800  2021-03-07 00:55:00+0800, 0, [], Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

# Reuse metric_df already loaded in the kernel.
# Ensure timezone object
tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 per (cmdb_id, kpi_name) using the entire metric_df
metric_thresholds = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'p95'})
)

# 2) Filter metric.csv to the target incident time window (inclusive)
start_ts = pd.to_datetime('2021-03-07 00:30:00+0800').tz_convert(tz)
end_ts = pd.to_datetime('2021-03-07 01:00:00+0800').tz_convert(tz)

# create tz-aware datetime column from epoch seconds (treat epoch as UTC then convert to Asia/Shanghai)
metric_df['_dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

metric_window = metric_df[(metric_df['_dt'] >= start_ts) & (metric_df['_dt'] <= end_ts)].copy()

# 3) For each (cmdb_id,kpi_name) present in metric_window compute required stats
# Merge p95 into window rows
metric_window_merged = metric_window.merge(metric_thresholds, how='left', on=['cmdb_id','kpi_name'])

# Define helper to compute per-group stats
def compute_group_stats(g):
    total_points = int(g.shape[0])
    p95 = float(g['p95'].iloc[0]) if pd.notnull(g['p95'].iloc[0]) else np.nan
    # anomaly rows where value >= p95 (use >= as requested). If p95 is NaN, anomaly_count = 0
    if np.isnan(p95):
        anomaly_mask = np.zeros(len(g), dtype=bool)
    else:
        anomaly_mask = g['value'] >= p95
    anomaly_count = int(anomaly_mask.sum())
    if anomaly_count > 0:
        earliest_anom_dt = g.loc[anomaly_mask, '_dt'].min()
        earliest_anom_str = pd.to_datetime(earliest_anom_dt).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')
    else:
        earliest_anom_str = pd.NA
    max_value = float(g['value'].max())
    severity_ratio = float(max_value / p95) if (pd.notnull(p95) and p95 > 0) else pd.NA
    return pd.Series({
        'cmdb_id': g['cmdb_id'].iloc[0],
        'kpi_name': g['kpi_name'].iloc[0],
        'total_points': total_points,
        'anomaly_count': anomaly_count,
        'earliest_anomaly_timestamp': earliest_anom_str,
        'max_value': max_value,
        'p95': p95,
        'severity_ratio': severity_ratio
    })

metric_anomalies = (
    metric_window_merged
    .groupby(['cmdb_id','kpi_name'], dropna=False)
    .apply(compute_group_stats)
    .reset_index(drop=True)
)

# 4) Prepare compact textual table: top 20 rows ordered by anomaly_count desc
metric_anomalies_sorted = metric_anomalies.sort_values(by='anomaly_count', ascending=False)
metric_anomalies_top20 = metric_anomalies_sorted[['cmdb_id','kpi_name','anomaly_count','total_points','earliest_anomaly_timestamp','severity_ratio']].head(20).reset_index(drop=True)

# Count how many pairs have anomaly_count > 0
anomaly_pairs_count = int((metric_anomalies['anomaly_count'] > 0).sum())

# Display compact results (variables) for the controller to consume
metric_thresholds, metric_window, metric_anomalies_top20, anomaly_pairs_count

```
Out[2]:
```
# Summarize results from previous computations into a single plain-English string.
# Reuse kernel variables: metric_thresholds, metric_window, metric_anomalies_top20, anomaly_pairs_count

top_lines = []
for _, r in metric_anomalies_top20.head(10).iterrows():
    top_lines.append(f"{r['cmdb_id']},{r['kpi_name']},{int(r['anomaly_count'])},{int(r['total_points'])},{r['earliest_anomaly_timestamp']},{r['severity_ratio']}")

summary = (
    f"Summary of metric anomaly analysis:\n\n"
    f"- metric_thresholds: computed P95 for {metric_thresholds.shape[0]} distinct (cmdb_id,kpi_name) pairs.\n"
    f"- metric_window: {metric_window.shape[0]} rows inside 2021-03-07 00:30:00+08:00 to 2021-03-07 01:00:00+08:00 (inclusive).\n"
    f"- metric_anomalies: computed per (cmdb_id,kpi_name) present in the window.\n"
    f"- Pairs with anomaly_count > 0: {anomaly_pairs_count} pairs.\n\n"
    f"Top anomaly candidates (up to 10) as cmdb_id,kpi_name,anomaly_count,total_points,earliest_anomaly_timestamp,severity_ratio:\n"
    + "\n".join(top_lines) + "\n\n"
    "Notes:\n"
    "- anomaly_count counts rows in the window where value >= the global P95 for that pair.\n"
    "- severity_ratio = max_value / p95 (shown as <NA> when p95 is zero or missing).\n"
    "- Many top pairs show anomaly_count = total_points (31), with earliest anomalies at 2021-03-07 00:30:00+0800.\n"
    "End of summary."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                           kpi_name           p95
0         IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09
1         IG01         JVM-Memory_7778_JVM_Memory_HeapMemoryUsage  3.846905e+01
2         IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  8.063301e+08
3         IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.796862e+07
4         IG01          JVM-Operating System_7778_JVM_JVM_CPULoad  1.469500e-01
...        ...                                                ...           ...
1703  dockerB2  Container-DOCKER_CONTAINER_2d16f5b2e830--bcou-...  1.280000e-01
1704  dockerB2  Container-DOCKER_CONTAINER_2d16f5b2e830--bcou-...  3.072000e+09
1705  dockerB2  Container-DOCKER_CONTAINER_2d16f5b2e830--bcou-...  7.572000e+01
1706  dockerB2  Container-DOCKER_CONTAINER_2d16f5b2e830--bcou-...  2.410754e+09
1707  dockerB2  Container-DOCKER_CONTAINER_2d16f5b2e830--bcou-...  2.167649e+05

[1708 rows x 3 columns],         timestamp   cmdb_id                                           kpi_name         value                       _dt
0      1615048200      IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09 2021-03-07 00:30:00+08:00
1      1615048200      IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  6.604902e+08 2021-03-07 00:30:00+08:00
2      1615048200      IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.796862e+07 2021-03-07 00:30:00+08:00
3      1615048200      IG01                    JVM-Runtime_7778_JVM_JVM_Uptime  1.272450e+05 2021-03-07 00:30:00+08:00
4      1615048200      IG01           JVM-Threads_7778_JVM_ThreadCount_Threads  7.300000e+01 2021-03-07 00:30:00+08:00
...           ...       ...                                                ...           ...                       ...
28448  1615050000  apache02                       OSLinux-OSLinux_SWAP_SWAP_Si  0.000000e+00 2021-03-07 01:00:00+08:00
28449  1615050000  apache02                       OSLinux-OSLinux_SWAP_SWAP_So  0.000000e+00 2021-03-07 01:00:00+08:00
28450  1615050000  apache02                 OSLinux-OSLinux_ZABBIX_Host_Uptime  6.248664e+06 2021-03-07 01:00:00+08:00
28451  1615050000  dockerA1  Container-DOCKER_CONTAINER_b6760337dc49--bcou-...  1.700000e-01 2021-03-07 01:00:00+08:00
28452  1615050000  dockerA2  Container-DOCKER_CONTAINER_94eca4f96efe--bcou-...  1.107700e+04 2021-03-07 01:00:00+08:00

[28453 rows x 5 columns],      cmdb_id                                           kpi_name  anomaly_count  total_points earliest_anomaly_timestamp severity_ratio
0    Redis01          redis-Redis_6379_Redis  (pubsub_channels)             31            31   2021-03-07 00:30:00+0800           <NA>
1   apache02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT             31            31   2021-03-07 00:30:00+0800           <NA>
2       IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax             31            31   2021-03-07 00:30:00+0800            1.0
3    Redis01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount             31            31   2021-03-07 00:30:00+0800            1.0
4    Mysql02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount             31            31   2021-03-07 00:30:00+0800            1.0
5    Redis01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT             31            31   2021-03-07 00:30:00+0800           <NA>
6    Redis01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT             31            31   2021-03-07 00:30:00+0800           <NA>
7    Redis01              redis-Redis_6379_Redis  (aof_enabled)             31            31   2021-03-07 00:30:00+0800           <NA>
8    Redis01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies             31            31   2021-03-07 00:30:00+0800            1.0
9    Mysql02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies             31            31   2021-03-07 00:30:00+0800           <NA>
10  apache01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies             31            31   2021-03-07 00:30:00+0800           <NA>
11   Mysql02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT             31            31   2021-03-07 00:30:00+0800           <NA>
12   Redis01                      redis-Redis_6379_redis server             31            31   2021-03-07 00:30:00+0800            1.0
13   Redis01             redis-Redis_6379_Redis  (expired_keys)             31            31   2021-03-07 00:30:00+0800            1.0
14      IG01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT             31            31   2021-03-07 00:30:00+0800           <NA>
15  apache02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount             31            31   2021-03-07 00:30:00+0800            1.0
16      IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax             31            31   2021-03-07 00:30:00+0800            1.0
17  apache02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies             31            31   2021-03-07 00:30:00+0800           <NA>
18  apache02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...             31            31   2021-03-07 00:30:00+0800            1.0
19  Tomcat02  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...             31            31   2021-03-07 00:30:00+0800           <NA>, 1708)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz
import numpy as np

# Reuse metric_anomalies from kernel
tz = pytz.timezone('Asia/Shanghai')

# Ensure metric_anomalies exists
# metric_anomalies should have columns: cmdb_id,kpi_name,anomaly_count,total_points,earliest_anomaly_timestamp,severity_ratio
# Coerce types
ma = metric_anomalies.copy()

# Normalize earliest_anomaly_timestamp to datetime (tz-aware) for accurate min computation
ma['_earliest_dt'] = pd.to_datetime(ma['earliest_anomaly_timestamp'], errors='coerce', utc=True).dt.tz_convert(tz)

# Normalize severity_ratio to numeric (NaN for missing)
ma['_severity'] = pd.to_numeric(ma['severity_ratio'], errors='coerce')

# Filter to KPIs with anomaly_count > 0 for aggregation
ma_pos = ma[ma['anomaly_count'] > 0].copy()

# 1) Aggregate by cmdb_id
component_aggr = (
    ma_pos
    .groupby('cmdb_id', dropna=False)
    .agg(
        anomalous_kpi_count = ('kpi_name', 'nunique'),           # count of KPIs with anomaly_count>0
        total_anomaly_points = ('anomaly_count', 'sum'),
        earliest_dt = ('_earliest_dt', 'min'),
        max_severity = ('_severity', 'max')
    )
    .reset_index()
)

# Format earliest_dt back to string with timezone, and convert max_severity NaN -> <NA>
def fmt_dt(x):
    if pd.isna(x):
        return pd.NA
    return pd.to_datetime(x).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')

component_aggr['earliest_anomaly_timestamp'] = component_aggr['earliest_dt'].apply(fmt_dt)
component_aggr['max_severity_ratio'] = component_aggr['max_severity'].where(~component_aggr['max_severity'].isna(), pd.NA)

# Select final columns and store full summary
component_anomaly_summary = component_aggr[['cmdb_id','anomalous_kpi_count','total_anomaly_points','earliest_anomaly_timestamp','max_severity_ratio']].copy()

# 2) Top 10 components ordered by anomalous_kpi_count desc, tie-breaker total_anomaly_points desc
component_anomaly_summary_top10 = component_anomaly_summary.sort_values(
    by=['anomalous_kpi_count','total_anomaly_points'],
    ascending=[False, False]
).head(10).reset_index(drop=True)

# 3) For top 3 components produce KPI-level details (up to 10 KPI rows per component)
top3_components = component_anomaly_summary_top10.head(3)['cmdb_id'].tolist()

# Filter metric_anomalies for these components and anomaly_count>0
kpi_details_list = []
for comp in top3_components:
    comp_kpis = ma_pos[ma_pos['cmdb_id'] == comp].copy()
    # Order by anomaly_count desc then severity desc
    comp_kpis_sorted = comp_kpis.sort_values(by=['anomaly_count','_severity'], ascending=[False, False])
    comp_topk = comp_kpis_sorted.head(10)[['cmdb_id','kpi_name','anomaly_count','total_points','earliest_anomaly_timestamp','severity_ratio']].copy()
    kpi_details_list.append(comp_topk)

if kpi_details_list:
    component_topkpi_details = pd.concat(kpi_details_list, ignore_index=True)
else:
    # empty DataFrame with expected columns
    component_topkpi_details = pd.DataFrame(columns=['cmdb_id','kpi_name','anomaly_count','total_points','earliest_anomaly_timestamp','severity_ratio'])

# Limit total rows to at most 30 just in case
component_topkpi_details = component_topkpi_details.head(30)

# 4) Scalar: how many distinct components have anomalous_kpi_count > 0
anomalous_components_count = int((component_anomaly_summary['anomalous_kpi_count'] > 0).sum())

# Return compact results: top10 summary, KPI details (up to 30 rows), and scalar count
component_anomaly_summary_top10, component_topkpi_details, anomalous_components_count

```
Out[3]:
```
# Summarize component and KPI-level anomaly results in plain English as a single string.
# Reuse kernel variables: component_anomaly_summary_top10, component_topkpi_details, anomalous_components_count

lines = []
lines.append("Component-level anomaly summary (top 10):")
lines.append("cmdb_id,anomalous_kpi_count,total_anomaly_points,earliest_anomaly_timestamp,max_severity_ratio")
for _, r in component_anomaly_summary_top10.iterrows():
    lines.append(f"{r['cmdb_id']},{int(r['anomalous_kpi_count'])},{int(r['total_anomaly_points'])},{r['earliest_anomaly_timestamp']},{r['max_severity_ratio']}")

lines.append("")    
lines.append(f"Distinct components with anomalous_kpi_count>0: {anomalous_components_count}")
lines.append("")

# KPI-level breakdown for top 3 components
top3 = component_anomaly_summary_top10.head(3)['cmdb_id'].tolist()
lines.append("KPI-level breakdown for top 3 components (up to 10 KPIs each):")
lines.append("cmdb_id,kpi_name,anomaly_count,total_points,earliest_anomaly_timestamp,severity_ratio")

# Use the prepared component_topkpi_details (already limited to <=30 rows)
if component_topkpi_details.shape[0] == 0:
    lines.append("<no KPI details available>")
else:
    # iterate rows in order (they are grouped by top component order)
    for _, r in component_topkpi_details.iterrows():
        # ensure proper formatting and compactness
        lines.append(f"{r['cmdb_id']},{r['kpi_name']},{int(r['anomaly_count'])},{int(r['total_points'])},{r['earliest_anomaly_timestamp']},{r['severity_ratio']}")

summary = "\n".join(lines)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id  anomalous_kpi_count  total_anomaly_points earliest_anomaly_timestamp  max_severity_ratio
0   Mysql02                  226                  2317   2021-03-07 00:30:00+0800            8.135869
1   Mysql01                  225                  2604   2021-03-07 00:30:00+0800            3.333333
2   Redis01                  113                  1347   2021-03-07 00:30:00+0800            4.670882
3  Tomcat04                  111                  1024   2021-03-07 00:30:00+0800            2.000000
4   Redis02                  109                  1434   2021-03-07 00:30:00+0800            5.846763
5  Tomcat02                  109                  1049   2021-03-07 00:30:00+0800            3.843634
6  Tomcat03                  109                  1022   2021-03-07 00:30:00+0800           10.000000
7  Tomcat01                  107                   953   2021-03-07 00:30:00+0800            3.204169
8      MG01                   92                   579   2021-03-07 00:30:00+0800            2.160552
9      MG02                   92                   544   2021-03-07 00:30:00+0800            2.000000,     cmdb_id                                        kpi_name  anomaly_count  total_points earliest_anomaly_timestamp severity_ratio
0   Mysql02      OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount             31            31   2021-03-07 00:30:00+0800            1.0
1   Mysql02  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT             31            31   2021-03-07 00:30:00+0800           <NA>
2   Mysql02    OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT             31            31   2021-03-07 00:30:00+0800           <NA>
3   Mysql02   OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies             31            31   2021-03-07 00:30:00+0800           <NA>
4   Mysql02          Mysql-MySQL_3306_Binlog stmt cache use             21            21   2021-03-07 00:30:00+0800            1.0
..      ...                                             ...            ...           ...                        ...            ...
25  Redis01      redis-Redis_6379_Redis  (used_memory_peak)             31            31   2021-03-07 00:30:00+0800            1.0
26  Redis01                   redis-Redis_6379_redis server             31            31   2021-03-07 00:30:00+0800            1.0
27  Redis01  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT             31            31   2021-03-07 00:30:00+0800           <NA>
28  Redis01    OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT             31            31   2021-03-07 00:30:00+0800           <NA>
29  Redis01           redis-Redis_6379_Redis  (aof_enabled)             31            31   2021-03-07 00:30:00+0800           <NA>

[30 rows x 6 columns], 29)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# Reuse trace_df already loaded in the kernel
tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 per (cmdb_id, trace_name) across full trace_df
trace_thresholds = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'p95'})
)

# 2) Filter trace_df to the incident time window (inclusive)
start_ts = pd.to_datetime('2021-03-07 00:30:00+0800').tz_convert(tz)
end_ts   = pd.to_datetime('2021-03-07 01:00:00+0800').tz_convert(tz)

# create tz-aware datetime column from epoch seconds
trace_df['_dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
trace_window = trace_df[(trace_df['_dt'] >= start_ts) & (trace_df['_dt'] <= end_ts)].copy()

# 3) For each (cmdb_id,trace_name) present in trace_window compute required stats
# Merge global p95 into window rows
trace_window_merged = trace_window.merge(trace_thresholds, how='left', on=['cmdb_id','trace_name'])

def compute_trace_stats(g):
    total_points = int(g.shape[0])
    p95 = float(g['p95'].iloc[0]) if pd.notnull(g['p95'].iloc[0]) else np.nan
    if np.isnan(p95):
        anomaly_mask = np.zeros(len(g), dtype=bool)
    else:
        anomaly_mask = g['value'] >= p95
    anomaly_count = int(anomaly_mask.sum())
    if anomaly_count > 0:
        earliest_dt = g.loc[anomaly_mask, '_dt'].min()
        earliest_str = pd.to_datetime(earliest_dt).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')
    else:
        earliest_str = pd.NA
    max_value = float(g['value'].max())
    severity_ratio = float(max_value / p95) if (pd.notnull(p95) and p95 > 0) else pd.NA
    return pd.Series({
        'cmdb_id': g['cmdb_id'].iloc[0],
        'trace_name': g['trace_name'].iloc[0],
        'total_points': total_points,
        'anomaly_count': anomaly_count,
        'earliest_anomaly_timestamp': earliest_str,
        'max_value': max_value,
        'p95': p95,
        'severity_ratio': severity_ratio
    })

trace_anomalies = (
    trace_window_merged
    .groupby(['cmdb_id','trace_name'], dropna=False)
    .apply(compute_trace_stats)
    .reset_index(drop=True)
)

# 4) Prepare compact top-20 table ordered by anomaly_count desc and scalar count
trace_anomalies_sorted = trace_anomalies.sort_values(by='anomaly_count', ascending=False)
trace_anomalies_top20 = trace_anomalies_sorted[['cmdb_id','trace_name','anomaly_count','total_points','earliest_anomaly_timestamp','severity_ratio']].head(20).reset_index(drop=True)

trace_anomalous_pairs_count = int((trace_anomalies['anomaly_count'] > 0).sum())

# For compactness in output, return:
# - trace_thresholds.shape (size summary) to avoid dumping full thresholds,
# - trace_window.shape (size of window),
# - trace_anomalies_top20 (compact table),
# - trace_anomalous_pairs_count (scalar)
trace_thresholds.shape, trace_window.shape, trace_anomalies_top20, trace_anomalous_pairs_count

```
Out[4]:
```
```python
# Build a plain-English summary string from existing kernel variables:
# trace_thresholds, trace_window, trace_anomalies_top20, trace_anomalous_pairs_count

lines = []
lines.append("Trace anomaly analysis summary:")
lines.append(f"- Global P95 thresholds computed for {trace_thresholds.shape[0]} distinct (cmdb_id,trace_name) pairs.")
lines.append(f"- Trace window rows (2021-03-07 00:30+08:00 through 01:00+08:00): {trace_window.shape[0]} rows.")
lines.append(f"- Number of (cmdb_id,trace_name) pairs with anomaly_count > 0: {trace_anomalous_pairs_count}.")
lines.append("")
lines.append("Top anomaly candidates (cmdb_id,trace_name,anomaly_count,total_points,earliest_anomaly_timestamp,severity_ratio):")

# Add up to top 10 rows from trace_anomalies_top20 in CSV-like compact form
for _, r in trace_anomalies_top20.head(10).iterrows():
    lines.append(f"{r['cmdb_id']},{r['trace_name']},{int(r['anomaly_count'])},{int(r['total_points'])},{r['earliest_anomaly_timestamp']},{r['severity_ratio']}")

lines.append("")
lines.append("Notes:")
lines.append("- anomaly_count counts rows in the incident window where value >= the global P95 for that (cmdb_id,trace_name).")
lines.append("- severity_ratio = max_value / p95 (shown as NA when p95 is zero or missing).")
lines.append("- Many top entries show multiple anomalies clustered around ~00:31–00:43+08:00 in the window.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

((228, 3), (6498, 5),      cmdb_id                       trace_name  anomaly_count  total_points earliest_anomaly_timestamp  severity_ratio
0       MG02    trace.from_dockerA1.row_count              4            24   2021-03-07 00:43:00+0800        1.222222
1   dockerA1          trace.to_MG02.row_count              4            24   2021-03-07 00:43:00+0800        1.222222
2       MG02    trace.from_Tomcat03.row_count              3            26   2021-03-07 00:31:00+0800        1.105263
3       IG02      trace.to_Tomcat01.row_count              3            30   2021-03-07 00:34:00+0800        1.475000
4   Tomcat03          trace.self.duration_p95              3            30   2021-03-07 00:31:00+0800        1.085714
5   Tomcat03          trace.to_MG02.row_count              3            26   2021-03-07 00:31:00+0800        1.105263
6   Tomcat01        trace.from_IG02.row_count              3            30   2021-03-07 00:34:00+0800        1.475000
7   dockerB1          trace.self.duration_p95              3            30   2021-03-07 00:37:00+0800        1.000000
8       IG01  trace.to_Tomcat01.duration_mean              2            30   2021-03-07 00:32:00+0800        1.243697
9       IG01   trace.to_Tomcat01.duration_p95              2            30   2021-03-07 00:32:00+0800        1.089258
10      IG01  trace.to_Tomcat02.duration_mean              2            30   2021-03-07 00:35:00+0800        4.023811
11      IG01   trace.to_Tomcat02.duration_p95              2            30   2021-03-07 00:35:00+0800        1.549549
12      IG01  trace.to_Tomcat04.duration_mean              2            30   2021-03-07 00:34:00+0800        1.272192
13      IG01   trace.to_Tomcat04.duration_p95              2            30   2021-03-07 00:33:00+0800        1.292676
14      IG01      trace.to_Tomcat04.row_count              2            30   2021-03-07 00:31:00+0800        1.301939
15      IG01      trace.to_Tomcat01.row_count              2            30   2021-03-07 00:36:00+0800        1.115435
16      IG01      trace.to_Tomcat02.row_count              2            30   2021-03-07 00:33:00+0800        1.202046
17      IG01  trace.to_Tomcat03.duration_mean              2            30   2021-03-07 00:34:00+0800        3.212720
18      IG01   trace.to_Tomcat03.duration_p95              2            30   2021-03-07 00:33:00+0800        1.456413
19      IG01      trace.to_Tomcat03.row_count              2            30   2021-03-07 00:33:00+0800        1.142857, 228)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing kernel variables: log_df, tz (if tz not present, create)
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 per (cmdb_id, log_name)
log_thresholds = (
    log_df
    .groupby(['cmdb_id', 'log_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'p95'})
)

# 2) Filter log_df to the incident time window (inclusive)
start_ts = pd.to_datetime('2021-03-07 00:30:00+0800').tz_convert(tz)
end_ts   = pd.to_datetime('2021-03-07 01:00:00+0800').tz_convert(tz)

# Ensure tz-aware datetime column on log_df (reuse if exists)
log_df['_dt'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
log_window = log_df[(log_df['_dt'] >= start_ts) & (log_df['_dt'] <= end_ts)].copy()

# 3) For each (cmdb_id,log_name) present in log_window compute required stats
log_window_merged = log_window.merge(log_thresholds, how='left', on=['cmdb_id','log_name'])

def compute_log_stats(g):
    total_points = int(g.shape[0])
    p95 = float(g['p95'].iloc[0]) if pd.notnull(g['p95'].iloc[0]) else np.nan
    if np.isnan(p95):
        anomaly_mask = np.zeros(len(g), dtype=bool)
    else:
        anomaly_mask = g['value'] >= p95
    anomaly_count = int(anomaly_mask.sum())
    if anomaly_count > 0:
        earliest_dt = g.loc[anomaly_mask, '_dt'].min()
        earliest_str = pd.to_datetime(earliest_dt).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')
    else:
        earliest_str = pd.NA
    max_value = float(g['value'].max())
    severity_ratio = float(max_value / p95) if (pd.notnull(p95) and p95 > 0) else pd.NA
    return pd.Series({
        'cmdb_id': g['cmdb_id'].iloc[0],
        'log_name': g['log_name'].iloc[0],
        'total_points': total_points,
        'anomaly_count': anomaly_count,
        'earliest_anomaly_timestamp': earliest_str,
        'max_value': max_value,
        'p95': p95,
        'severity_ratio': severity_ratio
    })

log_anomalies = (
    log_window_merged
    .groupby(['cmdb_id','log_name'], dropna=False)
    .apply(compute_log_stats)
    .reset_index(drop=True)
)

# 4) Aggregate log_anomalies by cmdb_id
la = log_anomalies.copy()
la['_severity'] = pd.to_numeric(la['severity_ratio'], errors='coerce')
la['_earliest_dt'] = pd.to_datetime(la['earliest_anomaly_timestamp'], errors='coerce', utc=True).dt.tz_convert(tz)

la_pos = la[la['anomaly_count'] > 0].copy()

component_log_summary = (
    la_pos
    .groupby('cmdb_id', dropna=False)
    .agg(
        anomalous_log_count = ('log_name', 'nunique'),
        total_log_anomaly_points = ('anomaly_count', 'sum'),
        earliest_log_dt = ('_earliest_dt', 'min'),
        max_log_severity_ratio = ('_severity', 'max')
    )
    .reset_index()
)

def fmt_dt(x):
    if pd.isna(x):
        return pd.NA
    return pd.to_datetime(x).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')

component_log_summary['earliest_log_anomaly_timestamp'] = component_log_summary['earliest_log_dt'].apply(fmt_dt)
component_log_summary['max_log_severity_ratio'] = component_log_summary['max_log_severity_ratio'].where(~component_log_summary['max_log_severity_ratio'].isna(), pd.NA)

component_log_summary = component_log_summary[['cmdb_id','anomalous_log_count','total_log_anomaly_points','earliest_log_anomaly_timestamp','max_log_severity_ratio']]

# 5) Prepare compact textual outputs
log_anomalies_top20 = log_anomalies.sort_values(by='anomaly_count', ascending=False)[['cmdb_id','log_name','anomaly_count','total_points','earliest_anomaly_timestamp','severity_ratio']].head(20).reset_index(drop=True)

component_log_summary_top10 = component_log_summary.sort_values(by=['anomalous_log_count','total_log_anomaly_points'], ascending=[False, False]).head(10).reset_index(drop=True)

log_anomalous_pairs_count = int((log_anomalies['anomaly_count'] > 0).sum())

# Return compact results: shapes for thresholds/window plus top tables and scalar
log_thresholds.shape, log_window.shape, log_anomalies_top20, component_log_summary_top10, log_anomalous_pairs_count

```
Out[5]:
```
- Global log thresholds: computed P95 for 18 distinct (cmdb_id, log_name) pairs.
- Incident window: 2021-03-07 00:30:00+08:00 through 2021-03-07 01:00:00+08:00 contains 394 log rows.
- Distinct (cmdb_id,log_name) pairs with anomaly_count > 0: 18.

Top log anomalies (cmdb_id, log_name, anomaly_count, total_points, earliest_anomaly_timestamp, severity_ratio):
- Tomcat03, log.error_count, 31, 31, 2021-03-07 00:30:00+0800, <NA>
- Tomcat02, log.error_count, 31, 31, 2021-03-07 00:30:00+0800, <NA>
- apache02,  log.error_count, 31, 31, 2021-03-07 00:30:00+0800, <NA>
- Tomcat04, log.error_count, 31, 31, 2021-03-07 00:30:00+0800, <NA>
- apache01,  log.error_count, 31, 31, 2021-03-07 00:30:00+0800, <NA>
- Tomcat01, log.error_count, 30, 30, 2021-03-07 00:30:00+0800, <NA>
- Mysql01,   log.row_count, 2, 2, 2021-03-07 00:31:00+0800, 1.0
- Mysql01,   log.error_count, 2, 2, 2021-03-07 00:31:00+0800, <NA>
- Tomcat02, log.row_count, 2, 31, 2021-03-07 00:34:00+0800, 1.054545
- Tomcat01, log.row_count, 2, 30, 2021-03-07 00:35:00+0800, 1.181592
(remaining top-20 rows include other log.row_count and log.gc_pause_max anomalies; many error_count anomalies are full-window hits at 00:30+08)

Component-level summary (top 10 by anomalous_log_count):
- Tomcat02: anomalous_log_count=3, total_log_anomaly_points=34, earliest=2021-03-07 00:30:00+0800, max_severity_ratio=1.054545
- Tomcat03: anomalous_log_count=3, total_log_anomaly_points=34, earliest=2021-03-07 00:30:00+0800, max_severity_ratio=1.243590
- Tomcat04: anomalous_log_count=3, total_log_anomaly_points=34, earliest=2021-03-07 00:30:00+0800, max_severity_ratio=1.299363
- Tomcat01: anomalous_log_count=3, total_log_anomaly_points=33, earliest=2021-03-07 00:30:00+0800, max_severity_ratio=1.181592
- apache01: anomalous_log_count=2, total_log_anomaly_points=33, earliest=2021-03-07 00:30:00+0800, max_severity_ratio=1.273312
- apache02: anomalous_log_count=2, total_log_anomaly_points=33, earliest=2021-03-07 00:30:00+0800, max_severity_ratio=1.214521
- Mysql01: anomalous_log_count=2, total_log_anomaly_points=4,  earliest=2021-03-07 00:31:00+0800, max_severity_ratio=1.0

Notes / interpretation:
- Many log.error_count anomalies are present across Tomcat and Apache hosts (most show anomaly_count equal to the full per-host window count, with earliest hit at 00:30+08).
- Some log.row_count and log.gc_pause_max entries also exceed their P95 thresholds with modest severity ratios (~1.04–1.30).
- Several severity_ratio values are <NA> where P95 is zero or not meaningful, so ratio cannot be computed.
- Overall there are 18 anomalous (cmdb_id,log_name) pairs to investigate further.

The original code execution output of IPython Kernel is also provided below for reference:

((18, 3), (394, 5),      cmdb_id          log_name  anomaly_count  total_points earliest_anomaly_timestamp severity_ratio
0   Tomcat03   log.error_count             31            31   2021-03-07 00:30:00+0800           <NA>
1   Tomcat02   log.error_count             31            31   2021-03-07 00:30:00+0800           <NA>
2   apache02   log.error_count             31            31   2021-03-07 00:30:00+0800           <NA>
3   Tomcat04   log.error_count             31            31   2021-03-07 00:30:00+0800           <NA>
4   apache01   log.error_count             31            31   2021-03-07 00:30:00+0800           <NA>
5   Tomcat01   log.error_count             30            30   2021-03-07 00:30:00+0800           <NA>
6    Mysql01     log.row_count              2             2   2021-03-07 00:31:00+0800            1.0
7    Mysql01   log.error_count              2             2   2021-03-07 00:31:00+0800           <NA>
8   Tomcat02     log.row_count              2            31   2021-03-07 00:34:00+0800       1.054545
9   Tomcat01     log.row_count              2            30   2021-03-07 00:35:00+0800       1.181592
10  apache01     log.row_count              2            31   2021-03-07 00:34:00+0800       1.273312
11  Tomcat04     log.row_count              2            31   2021-03-07 00:34:00+0800       1.299363
12  apache02     log.row_count              2            31   2021-03-07 00:34:00+0800       1.214521
13  Tomcat03     log.row_count              2            31   2021-03-07 00:35:00+0800        1.24359
14  Tomcat03  log.gc_pause_max              1             5   2021-03-07 00:43:00+0800        1.09375
15  Tomcat02  log.gc_pause_max              1             8   2021-03-07 00:36:00+0800       1.040462
16  Tomcat01  log.gc_pause_max              1             3   2021-03-07 00:35:00+0800       1.066667
17  Tomcat04  log.gc_pause_max              1             4   2021-03-07 00:34:00+0800       1.081081,     cmdb_id  anomalous_log_count  total_log_anomaly_points earliest_log_anomaly_timestamp  max_log_severity_ratio
0  Tomcat02                    3                        34       2021-03-07 00:30:00+0800                1.054545
1  Tomcat03                    3                        34       2021-03-07 00:30:00+0800                1.243590
2  Tomcat04                    3                        34       2021-03-07 00:30:00+0800                1.299363
3  Tomcat01                    3                        33       2021-03-07 00:30:00+0800                1.181592
4  apache01                    2                        33       2021-03-07 00:30:00+0800                1.273312
5  apache02                    2                        33       2021-03-07 00:30:00+0800                1.214521
6   Mysql01                    2                         4       2021-03-07 00:31:00+0800                1.000000, 18)```
```

In [ ]:
# In[6]:

The Executor failed to complete the instruction, please re-write a new instruction for Executor.

```
Out[6]:
```
The Executor failed to complete the instruction, please re-write a new instruction for Executor.```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz
import re

# Reuse kernel variables: metric_anomalies, metric_thresholds, metric_window, component_anomaly_summary_top10
tz = pytz.timezone('Asia/Shanghai')

# Component list in order
comp_list = component_anomaly_summary_top10['cmdb_id'].tolist()

# Subset KPI rows for these components
ma_sub = metric_anomalies[metric_anomalies['cmdb_id'].isin(comp_list)].copy()

# Compute max_value per (cmdb_id,kpi_name) in metric_window
group_max = (
    metric_window
    .groupby(['cmdb_id','kpi_name'], dropna=False)['value']
    .max()
    .reset_index()
    .rename(columns={'value':'max_value'})
)

# Prepare thresholds
mt = metric_thresholds.copy()

# Merge thresholds and max into KPI rows
comp_kpi = ma_sub.merge(mt, how='left', on=['cmdb_id','kpi_name']).merge(group_max, how='left', on=['cmdb_id','kpi_name'])

# After merges there may be duplicate p95 columns (e.g., 'p95_x','p95_y'). Find first p95-like column and first max_value-like column
p95_cols = [c for c in comp_kpi.columns if re.fullmatch(r'p95(_x|_y|_\d+)?|p95', c)]
max_cols = [c for c in comp_kpi.columns if re.fullmatch(r'max_value(_x|_y|_\d+)?|max_value', c)]

# Choose columns if present
p95_col = p95_cols[0] if p95_cols else None
max_col = max_cols[0] if max_cols else None

# If metric_anomalies already had a p95 column, prefer it
if 'p95' in comp_kpi.columns:
    p95_col = 'p95'
# Ensure max_col exists
if max_col is None:
    # try to compute from metric_window as fallback
    gm = group_max.set_index(['cmdb_id','kpi_name'])['max_value']
    comp_kpi['max_value_fallback'] = comp_kpi.set_index(['cmdb_id','kpi_name']).index.map(gm)
    max_col = 'max_value_fallback'

# Normalize numeric columns
comp_kpi['anomaly_count'] = pd.to_numeric(comp_kpi['anomaly_count'], errors='coerce').fillna(0).astype(int)
comp_kpi['total_points'] = pd.to_numeric(comp_kpi['total_points'], errors='coerce').fillna(0).astype(int)

# Compute severity_ratio = max_value / p95 (null if p95 <= 0 or missing). Use identified columns
def compute_sev(row):
    pv = row.get(p95_col) if p95_col in row.index else np.nan
    mv = row.get(max_col) if max_col in row.index else np.nan
    try:
        if pd.isna(pv) or pv == 0:
            return pd.NA
        return float(mv) / float(pv)
    except Exception:
        return pd.NA

comp_kpi['severity_ratio_computed'] = comp_kpi.apply(compute_sev, axis=1)

# For sorting treat NA as smallest: create sort key where NA -> -inf, else value
comp_kpi['_severity_sort'] = comp_kpi['severity_ratio_computed'].apply(lambda x: -np.inf if pd.isna(x) else x)

# Sort per component by severity desc then anomaly_count desc
comp_kpi_sorted = comp_kpi.sort_values(by=['cmdb_id','_severity_sort','anomaly_count'], ascending=[True, False, False])

# For each component, take up to top 5 KPI rows in the given component order
rows = []
for comp in comp_list:
    sub = comp_kpi_sorted[comp_kpi_sorted['cmdb_id'] == comp]
    if sub.empty:
        continue
    topk = sub.head(5)
    rows.append(topk)

if rows:
    component_kpi_severity = pd.concat(rows, ignore_index=True)
else:
    component_kpi_severity = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','anomaly_count','total_points','earliest_anomaly_timestamp',
        'severity_ratio_computed', p95_col if p95_col else 'p95', max_col if max_col else 'max_value'
    ])

# Limit total rows to at most 50
component_kpi_severity = component_kpi_severity.head(50).copy()

# Select and rename final columns as requested
final_cols = ['cmdb_id','kpi_name','anomaly_count','total_points','earliest_anomaly_timestamp','severity_ratio_computed']
# Determine p95 and max_value column names present
p95_final = p95_col if p95_col in component_kpi_severity.columns else None
max_final = max_col if max_col in component_kpi_severity.columns else None

if p95_final:
    final_cols.append(p95_final)
else:
    component_kpi_severity['p95'] = pd.NA
    final_cols.append('p95')

if max_final:
    final_cols.append(max_final)
else:
    component_kpi_severity['max_value'] = pd.NA
    final_cols.append('max_value')

component_kpi_severity = component_kpi_severity[final_cols].rename(columns={
    'severity_ratio_computed':'severity_ratio',
    p95_final: 'p95' if p95_final else 'p95',
    max_final: 'max_value' if max_final else 'max_value'
})

# Ensure column order exactly as requested
component_kpi_severity = component_kpi_severity[[
    'cmdb_id','kpi_name','anomaly_count','total_points','earliest_anomaly_timestamp','severity_ratio','p95','max_value'
]]

# Return the compact table
component_kpi_severity.head(50)

```
Out[7]:
```
- I produced component_kpi_severity (up to top 5 KPIs per each of the top-10 components; table limited to 50 rows). Each row contains: cmdb_id, kpi_name, anomaly_count, total_points, earliest_anomaly_timestamp, severity_ratio, p95, max_value.

- Overall observation: many KPIs show anomalies starting around 2021-03-07 00:30–00:34+08:00.

- Highest severity examples (from the produced table):
  - Mysql02, Mysql-MySQL_3306_Innodb Row Lock Time — severity_ratio 8.135869 (p95 ≈ 0.9833, max_value = 8.0), earliest anomaly 2021-03-07 00:31:00+0800.
  - Mysql02, Mysql-MySQL_3306_Handler Rollback — severity_ratio 5.071856 (p95 ≈ 0.0167, max_value ≈ 0.0847), earliest 2021-03-07 00:34:00+0800.
  - Mysql02, OSLinux local disk / DSKBps & DSKRTps & DSKRead — multiple disk KPIs with severity_ratio up to 4.0 (p95 and max_value shown per row).
  - MG02 shows elevated disk I/O metrics (e.g., DSKRead: p95 ≈ 188.79, max ≈ 338.67, severity ≈ 1.79).
  - Several components (Mysql01, Redis01, Tomcats, MG01/MG02) also appear with KPIs having severity_ratio > 1 indicating values exceeded global P95.

- Interpretation guidance:
  - severity_ratio = max_value / p95; values >>1 indicate substantial exceedance of the historical 95th percentile.
  - The most critical KPIs to investigate first are the few with the largest severity_ratio (the Mysql02 Innodb Row Lock Time and Handler Rollback are top candidates).
  - Disk I/O and MySQL-related KPIs are prominent among top severity cases, suggesting IO/DB contention as likely areas to investigate.

- If you want, I can:
  - return the component_kpi_severity table (CSV-like) for direct inspection,
  - compute the single earliest anomaly timestamp across these top-10 components, or
  - drill into raw metric_window rows for any specific KPI for detailed timestamps/values.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  anomaly_count  total_points earliest_anomaly_timestamp severity_ratio         p95    max_value
0   Mysql02              Mysql-MySQL_3306_Innodb Row Lock Time              2            21   2021-03-07 00:31:00+0800       8.135869    0.983300     8.000000
1   Mysql02                  Mysql-MySQL_3306_Handler Rollback              2            21   2021-03-07 00:34:00+0800       5.071856    0.016700     0.084700
2   Mysql02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps              1            16   2021-03-07 00:52:00+0800            4.0    0.750000     3.000000
3   Mysql02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps              1            16   2021-03-07 00:30:00+0800            4.0    0.004167     0.016667
4   Mysql02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead              1            16   2021-03-07 00:30:00+0800            4.0    0.033333     0.133333
..      ...                                                ...            ...           ...                        ...            ...         ...          ...
45     MG02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT              2            21   2021-03-07 00:33:00+0800            2.0    2.000000     4.000000
46     MG02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...              1            11   2021-03-07 00:54:00+0800       1.968569  604.500000  1190.000000
47     MG02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps              1            12   2021-03-07 00:54:00+0800       1.956522    4.600000     9.000000
48     MG02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...              1            10   2021-03-07 00:32:00+0800       1.818182    0.055000     0.100000
49     MG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead              1            10   2021-03-07 00:32:00+0800       1.793912  188.786685   338.666700

[50 rows x 8 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing kernel variables: metric_window, metric_thresholds
tz = pytz.timezone('Asia/Shanghai')

# Target component and KPI list
comp = 'Mysql02'
kpis = [
    'Mysql-MySQL_3306_Innodb Row Lock Time',
    'Mysql-MySQL_3306_Handler Rollback',
    'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps',
    'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps',
    'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead'
]

# 1) Filter metric_window for the component and KPI names
mysql02_kpi_rows = metric_window[
    (metric_window['cmdb_id'] == comp) &
    (metric_window['kpi_name'].isin(kpis))
].copy()

# 2) Merge global p95 thresholds from metric_thresholds (precomputed globally)
mysql02_kpi_rows = mysql02_kpi_rows.merge(
    metric_thresholds,
    how='left',
    on=['cmdb_id', 'kpi_name']
)

# 3) Compute formatted timestamp string and is_anomaly flag (value >= p95 when p95 is present)
mysql02_kpi_rows['timestamp'] = pd.to_datetime(mysql02_kpi_rows['_dt']).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S%z')
# p95 column name from metric_thresholds is 'p95'
mysql02_kpi_rows['is_anomaly'] = np.where(
    mysql02_kpi_rows['p95'].notna() & (mysql02_kpi_rows['value'] >= mysql02_kpi_rows['p95']),
    True,
    False
)

# 4) Sort rows by kpi_name then timestamp ascending (so each KPI's rows are time-ordered)
mysql02_kpi_rows = mysql02_kpi_rows.sort_values(by=['kpi_name','_dt'], ascending=[True, True]).reset_index(drop=True)

# Store full extracted table in kernel variable (as required)
# mysql02_kpi_rows already assigned.

# 5) Prepare compact CSV-like view with requested columns, limited to at most 50 rows for display
compact_view = mysql02_kpi_rows[['kpi_name','timestamp','value','p95','is_anomaly']].head(50)

# Display the compact table (variable output)
compact_view

```
Out[8]:
```
I extracted metric_window rows for Mysql02 for the five requested KPIs, joined each row with its global P95, and computed an is_anomaly flag (value >= p95). The extracted table (mysql02_kpi_rows) contains 50 rows (the display was limited to 50).

Key findings from the extracted rows:
- A confirmed anomaly: Mysql-MySQL_3306_Handler Rollback exceeded its P95 at 2021-03-07 00:34:00+08:00 (value 0.0847 vs p95 0.0167 → is_anomaly = True).
- MySQL innodb row lock time (Mysql-MySQL_3306_Innodb Row Lock Time) is a strong candidate: earlier summary shows it exceeded P95 substantially (severity ≈ 8.14, max_value = 8.0, p95 ≈ 0.9833) with earliest anomaly around 2021-03-07 00:31:00+08:00 — this KPI should be investigated first.
- Disk I/O KPIs (OSLinux … DSKBps, DSKRTps, DSKRead) are included; some of these show values exceeding their P95 in the overall KPI-level summary (severity > 1), indicating intermittent disk I/O spikes during the window. The per-timestamp is_anomaly flags in the extracted rows let you see exactly which timestamps exceeded P95.

Next suggested steps:
- Investigate Mysql02 Innodb Row Lock Time and Handler Rollback occurrences around 00:30–00:35+08:00 for root cause (long queries, lock contention, or retries).
- Check disk I/O spikes on sda/sdb around the same timeframe as they can drive DB performance issues.

If you want, I can:
- Show the full mysql02_kpi_rows table (CSV-like) for inspection, or
- Filter to only anomalous rows across these KPIs for quick review.

The original code execution output of IPython Kernel is also provided below for reference:

kpi_name                 timestamp   value     p95  is_anomaly
0                Mysql-MySQL_3306_Handler Rollback  2021-03-07 00:30:00+0800  0.0000  0.0167       False
1                Mysql-MySQL_3306_Handler Rollback  2021-03-07 00:31:00+0800  0.0000  0.0167       False
2                Mysql-MySQL_3306_Handler Rollback  2021-03-07 00:33:00+0800  0.0164  0.0167       False
3                Mysql-MySQL_3306_Handler Rollback  2021-03-07 00:34:00+0800  0.0847  0.0167        True
4                Mysql-MySQL_3306_Handler Rollback  2021-03-07 00:36:00+0800  0.0000  0.0167       False
..                                             ...                       ...     ...     ...         ...
45  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps  2021-03-07 00:36:00+0800  0.0000  0.7500       False
46  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps  2021-03-07 00:38:00+0800  0.0000  0.7500       False
47  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps  2021-03-07 00:40:00+0800  0.0000  0.7500       False
48  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps  2021-03-07 00:42:00+0800  0.0000  0.7500       False
49  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps  2021-03-07 00:44:00+0800  0.0000  0.7500       False

[50 rows x 5 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing kernel variables: metric_window, metric_thresholds
tz = pytz.timezone('Asia/Shanghai')

# Parameters
comp = 'Mysql02'
kpis = [
    'Mysql-MySQL_3306_Innodb Row Lock Time',
    'Mysql-MySQL_3306_Handler Rollback'
]

# 1) Select rows within metric_window for the component & KPIs and join p95
mw_sub = metric_window[
    (metric_window['cmdb_id'] == comp) &
    (metric_window['kpi_name'].isin(kpis))
].copy()

# Merge global p95 thresholds (metric_thresholds computed globally earlier)
mw_sub = mw_sub.merge(metric_thresholds, how='left', on=['cmdb_id','kpi_name'])  # p95 column

# Ensure timestamp numeric and _dt tz-aware exist
mw_sub['timestamp'] = pd.to_numeric(mw_sub['timestamp'], errors='coerce').astype('Int64')
# _dt should already be tz-aware from earlier; ensure it exists
if '_dt' not in mw_sub.columns:
    mw_sub['_dt'] = pd.to_datetime(mw_sub['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Mark is_anomaly where value >= p95 (require p95 non-null and > =)
mw_sub['is_anomaly'] = np.where(mw_sub['p95'].notna() & (mw_sub['value'] >= mw_sub['p95']), True, False)

# 2) Keep only anomaly rows and group into consecutive-by-minute segments per KPI
anom = mw_sub[mw_sub['is_anomaly']].copy()

# If no anomaly rows, create empty result
if anom.empty:
    mysql02_fault_segments = pd.DataFrame(columns=[
        'kpi_name','segment_start_timestamp','segment_end_timestamp',
        'segment_length_in_minutes','segment_max_value','p95','segment_max_severity'
    ])
else:
    # Sort per KPI by timestamp
    anom = anom.sort_values(['kpi_name','timestamp'], ascending=[True, True]).reset_index(drop=True)

    # Compute gap (seconds) from previous row within same KPI
    anom['prev_ts'] = anom.groupby('kpi_name')['timestamp'].shift(1)
    anom['gap'] = (anom['timestamp'] - anom['prev_ts']).fillna(9999).astype(float)

    # Start new segment when gap > 60 seconds
    anom['new_segment'] = (anom['gap'] > 60).astype(int)
    # Cumulative segment id per KPI
    anom['segment_id'] = anom.groupby('kpi_name')['new_segment'].cumsum()

    # 2) Aggregate per KPI and segment_id
    segs = (
        anom
        .groupby(['kpi_name','segment_id'], dropna=False)
        .agg(
            segment_start_ts=('timestamp','min'),
            segment_end_ts=('timestamp','max'),
            segment_length_in_minutes=('timestamp','size'),
            segment_max_value=('value','max'),
            p95=('p95','first')  # p95 is same per KPI
        )
        .reset_index()
    )

    # Compute segment_max_severity (null if p95 missing or <=0)
    def compute_seg_sev(row):
        pv = row['p95']
        mv = row['segment_max_value']
        try:
            if pd.isna(pv) or pv == 0:
                return pd.NA
            return float(mv) / float(pv)
        except Exception:
            return pd.NA

    segs['segment_max_severity'] = segs.apply(compute_seg_sev, axis=1)

    # Format timestamps to tz-aware strings
    segs['segment_start_timestamp'] = pd.to_datetime(segs['segment_start_ts'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S%z')
    segs['segment_end_timestamp']   = pd.to_datetime(segs['segment_end_ts'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S%z')

    # Select and order columns as requested
    mysql02_fault_segments = segs[[
        'kpi_name',
        'segment_start_timestamp',
        'segment_end_timestamp',
        'segment_length_in_minutes',
        'segment_max_value',
        'p95',
        'segment_max_severity'
    ]].copy()

    # 3) Compute single earliest anomaly timestamp across both KPIs (min segment_start)
    if not mysql02_fault_segments.empty:
        earliest_overall = mysql02_fault_segments['segment_start_timestamp'].min()
    else:
        earliest_overall = pd.NA

    # Order segments by start time ascending and limit to at most 10 rows
    mysql02_fault_segments = mysql02_fault_segments.sort_values('segment_start_timestamp', ascending=True).head(10).reset_index(drop=True)

# Store full result in kernel variable (already assigned)
# Return only the compact table (CSV-like)
mysql02_fault_segments

```
Out[9]:
```
Summary of detected anomaly segments for Mysql02 (two KPIs):

- Earliest anomaly timestamp across both KPIs: 2021-03-07 00:31:00+08:00.

Detected anomaly segments (kpi_name, segment_start, segment_end, length(min), segment_max_value, p95, segment_max_severity):
1) Mysql-MySQL_3306_Innodb Row Lock Time, 2021-03-07 00:31:00+0800 → 2021-03-07 00:31:00+0800, 1 min, max=0.9833, p95=0.9833, severity=1.00
2) Mysql-MySQL_3306_Handler Rollback, 2021-03-07 00:34:00+0800 → 2021-03-07 00:34:00+0800, 1 min, max=0.0847, p95=0.0167, severity≈5.07
3) Mysql-MySQL_3306_Handler Rollback, 2021-03-07 00:37:00+0800 → 2021-03-07 00:37:00+0800, 1 min, max=0.0167, p95=0.0167, severity=1.00
4) Mysql-MySQL_3306_Innodb Row Lock Time, 2021-03-07 00:39:00+0800 → 2021-03-07 00:39:00+0800, 1 min, max=8.0000, p95=0.9833, severity≈8.14

Interpretation / next steps (concise):
- The most severe events are:
  - Innodb Row Lock Time at 00:39 (severity ≈ 8.14) — highest priority.
  - Handler Rollback at 00:34 (severity ≈ 5.07) — also high priority.
- These are short, discrete one-minute spikes. Investigate MySQL activity around 00:30–00:40+08:00 (long-running/blocked queries, lock contention, retries/rollbacks) and correlate with disk I/O and host metrics.

The original code execution output of IPython Kernel is also provided below for reference:

kpi_name   segment_start_timestamp     segment_end_timestamp  segment_length_in_minutes  segment_max_value     p95  segment_max_severity
0  Mysql-MySQL_3306_Innodb Row Lock Time  2021-03-07 00:31:00+0800  2021-03-07 00:31:00+0800                          1             0.9833  0.9833              1.000000
1      Mysql-MySQL_3306_Handler Rollback  2021-03-07 00:34:00+0800  2021-03-07 00:34:00+0800                          1             0.0847  0.0167              5.071856
2      Mysql-MySQL_3306_Handler Rollback  2021-03-07 00:37:00+0800  2021-03-07 00:37:00+0800                          1             0.0167  0.0167              1.000000
3  Mysql-MySQL_3306_Innodb Row Lock Time  2021-03-07 00:39:00+0800  2021-03-07 00:39:00+0800                          1             8.0000  0.9833              8.135869```
```